### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part6_autodiff_tests.py

import mugrade
import os
from part6_autodiff_tests import *
os.environ["MUGRADE_HW"] = "Part 6 - Automatic Differentiation"
os.environ["MUGRADE_KEY"] = "" ### Your key here

def _mugrade_name(name):
    def rename(function):
        function.__name__ = name
        return function
    return rename

### Download the necessary files
from huggingface_hub import hf_hub_download

filenames = [
    'config.d12.json',
    'llm.d12.pt',
    'fineweb-edu-10BT.shuffle.bin'
]
for f in filenames:
    if not os.path.exists(f):
        hf_hub_download(repo_id="zkolter/llm_speedrun", filename=f, repo_type="dataset", local_dir=".")


### Original LLM Architecture

In this first portion of the notebook, you'll copy over your existing implementation and save the gradients.  Specifically, copy the code for the LLM class (and support functions) as well as the `cross_entropy_loss` function below.

In [ ]:
### BEGIN YOUR CODE
pass
### END YOUR CODE

After you have done this, the following code will save the gradient of the cross entropy loss with respect to the model parameters for the first 4 sequences in the fineweb EDU dataset.

In [ ]:
## compute gradients for parameters on the first 8 batches in fineweb
from array import array

with open("fineweb-edu-10BT.shuffle.bin", "rb") as f:
    tokens = array("H", f.read(4*(2048+1)*2)).tolist()
tokens = torch.tensor(tokens).reshape(4,2049).cuda()

with open("config.d12.json", "rt") as f: config = json.load(f)
llm = LLM(config)
llm.load("llm.d12.pt")
for k,p in llm.params.items(): 
    llm.params[k] = p.detach().cuda().requires_grad_()
for k,p in llm.buffers.items(): 
    llm.buffers[k] = llm.buffers[k].cuda()

logits = llm(tokens[:,:-1]).float()
loss = cross_entropy_loss(logits, tokens[:,1:])
loss.backward()

grads = {k:p.grad for k,p in llm.params.items()}
torch.save(grads, "grads.pt")

### Automatic differentiation

Now, implement the automatic differentiation library presented in class.  The additional functions you will need to implement, besides those presented in the class, are the `silu()`, `cross_entropy_loss` and `rms_norm` functions.  You'll have to work out the implementation of these functions.

Local tests use small NumPy arrays and need no dataset or saved model. Tensor operation tests check the forward and vector-Jacobian-product callbacks registered with `Tensor.build`. Implement the Tensor engine before testing the adapted LLM. Uncomment `# @mugrade.local_tests` to test a function, or change it to `@mugrade.submit_tests` to submit it. Keep the `_mugrade_name` decorators: they give the two constructors distinct grading names. The copied PyTorch LLM above is not graded in this assignment.


In [ ]:
del torch
import cupy as np
import math

class Tensor:
    # @mugrade.local_tests
    @_mugrade_name("Tensor_init")
    def __init__(self, value, vjp=None, parents=None, requires_grad=False):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    @staticmethod
    # @mugrade.local_tests
    def build(forward, vjp, *args):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def __add__(self, other):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def __mul__(self, other):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def __matmul__(self, other):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def __getitem__(self, idx):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def __setitem__(self, idx, x):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def transpose(self, i, j):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def reshape(self, *shape):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def flip(self, dim):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def to(self, dtype):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE


    @property
    # @mugrade.local_tests
    def shape(self): 
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE
        
    @property
    # @mugrade.local_tests
    def dtype(self): 
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def backward(self):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE
        

# @mugrade.local_tests
def softmax(x):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


# @mugrade.local_tests
def cross_entropy_loss(logits, y):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


# @mugrade.local_tests
def silu(x):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


# @mugrade.local_tests
def rms_norm(x):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


# @mugrade.local_tests
def self_attn(q,k,v,mask):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE


        

Now, adapt the LLM implementation above to use your automatic differentiation engine.  You will want to ensure that none of the code uses torch, even to build the buffers.  Most of this should be exactly the same as your existing LLM implementation, and here we test only the functions that are different.  Note that you don't need to implement the `generate` or load/save functionality in this version of the LLM, as it won't be used directly.

In [ ]:
# @mugrade.local_tests
def embedding(x, weights, dtype):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

# @mugrade.local_tests
def linear(x, weights):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

class LLM:
    # @mugrade.local_tests
    def rope(self, x, pos=0):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def multihead_attn(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def mlp(self, x, layer):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def transformer_block(self, x, layer, mask, pos=0, cache=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE
    
    # @mugrade.local_tests
    @_mugrade_name("LLM_init")
    def __init__(self, config):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE
        
            
    # @mugrade.local_tests
    def __call__(self, tokens, pos=0, cache=None):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE


### Validate gradients

Finally, the following code will compute the same gradients as we did above with PyTorch, but using your own library.  If you did everything correctly, then these will be the same as the PyTorch gradients up to floating point precision.

In [ ]:
import json
import torch  # Used only to load the saved PyTorch tensors.
from array import array

with open("config.d12.json", "rt") as f: config = json.load(f)
llm = LLM(config)
params = torch.load("llm.d12.pt")
for k,v in params.items():
    llm.params[k] = Tensor(np.array(v.detach().cpu().numpy()), requires_grad=True)

## compute gradients for parameters on the first 8 batches in fineweb
with open("fineweb-edu-10BT.shuffle.bin", "rb") as f:
    tokens = array("H", f.read(4*(2048+1)*2)).tolist()
tokens = np.array(tokens).reshape(4,2049)

out = llm(tokens[:,:-1])
loss = cross_entropy_loss(out, tokens[:,1:])
loss.backward()

true_grads = torch.load("grads.pt")
for k,p in llm.params.items():
    print(k, np.linalg.norm(p.grad - np.array(true_grads[k].cpu().numpy())))